# Stillwater GWI/RDII Heterogeneity Project Sibling

This notebook creates a SewerTris project sibling from the completed Stillwater base model. The physical city layout, roads, DEM, sewer network, pipe design, and base hydraulic properties are reused. Only the dynamic SWMM input data are changed:

- GWI is assigned from a spatial raster of pipe inflow coefficients.
- RDII imperviousness is assigned from a spatial RDII density raster.

This is the project-style version of the idea explored in `Full_SewerTris_04.ipynb`, with the GWI raster applied directly to the scenario input that is actually simulated.

## Setup

Load the local refactored package and the completed Stillwater base project.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import rasterio

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "sewertris").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXAMPLES_DIR = PROJECT_ROOT / "examples"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import sewertris

for _module_name in [
    "sewertris._deps",
    "sewertris.domain",
    "sewertris.layout",
    "sewertris.roads",
    "sewertris.topography",
    "sewertris.sewer_network",
    "sewertris.hydrology",
    "sewertris.design",
    "sewertris.swmm",
    "sewertris.plots",
    "sewertris.project",
    "sewertris",
]:
    if _module_name in sys.modules:
        importlib.reload(sys.modules[_module_name])

import sewertris as sp

base_project_path = EXAMPLES_DIR / "output_example_2_project"

base_project = sp.SewerTrisProject.load(base_project_path)
print("Base project:", base_project.project_file)


## Define Sibling

The `changes` dictionary only touches Step 10 dynamic inputs, so the sibling reuses the completed physical system and rebuilds the SWMM scenario with heterogeneous GWI/RDII inputs.

In [ ]:
heterogeneity_changes = {
    "gwi_raster": {
        "min_value": 0.00001,
        "max_value": 0.00005,
        "random_seed": 42,
        "samples_per_pipe": 5,
    },
    "rdii_raster": {
        "min_density": 0.0,
        "max_density": 5.0,
        "random_seed": 123,
        "rdii_to_imperv_scale": (0.0, 300.0),
    },
    "rdii": {
        "infiltration_params": (50, 0.5, 7, "", ""),
        "width": 100,
        "slope": 0.005,
        "curblen": 0,
    },
}

project = base_project.clone_sibling(
    EXAMPLES_DIR / "output_example_4_project",
    name="Stillwater GWI/RDII Heterogeneity Project",
    changes=heterogeneity_changes,
)

print("Sibling project:", project.project_file)
print("Rerun from:", project.metadata["lineage"]["rerun_from"])
print("Copied artifacts:", project.metadata["lineage"]["copied_artifacts"])


## Run Sibling

Rerun the dynamic-input workflow using parent parameters plus the raster heterogeneity changes. The GWI raster and RDII raster are generated inside `output_example_4_project` and applied to the scenario SWMM file in-place.

In [ ]:
results = project.rerun_from_parent_parameters(
    base_project,
    scenario_name="bwf_gwi_rdii",
    run_flow_components=True,
)

scenario = results["scenario"]
df = results["flow_components"]

print("GWI raster:", project.gwi_raster_path)
print("RDII raster:", project.rdii_raster_path)
print("Scenario input:", scenario.swmm_inp_path)
print("Flow components:", scenario.flows_path)


## Inspect Heterogeneity Rasters

These rasters are the only intended model input changes relative to the Stillwater base project.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, raster_path, title, label in [
    (axes[0], project.gwi_raster_path, "GWI inflow coefficient raster", "L/s/m"),
    (axes[1], project.rdii_raster_path, "RDII density raster", "RDII density"),
]:
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        bounds = src.bounds
    im = ax.imshow(
        data,
        extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
        origin="upper",
        cmap="viridis",
    )
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Easting")
    ax.set_ylabel("Northing")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=label)

plt.tight_layout()
plt.show()


## Flow Components

Plot the decomposed hydrograph for the heterogeneous GWI/RDII scenario.

In [ ]:
options = base_project.step_parameters("10_dynamic_flow_input_definition_base_model").get("options_dict", {})
start = f"{options.get('START_DATE', '01/01/1990')} {options.get('START_TIME', '00:00:00')}"
end = f"{options.get('END_DATE', '01/10/1990')} {options.get('END_TIME', '00:00:00')}"

sp.plot_flow_components_v2(
    df,
    start=start,
    end=end,
    title="Stillwater heterogeneous GWI/RDII flow components",
)
project.save()
print("Project metadata:", project.project_file)


## Compare Base vs Heterogeneous Inputs

Because the physical system is reused, the most informative comparison plots are the SWMM inflow map and the flow-component hydrograph.

In [ ]:
sp.plot_two_models(
    "inflow",
    base_project,
    project,
    labels=("Original Stillwater", "Raster GWI/RDII"),
    scenario_name="bwf_gwi_rdii",
)

sp.plot_two_models(
    "flow_components",
    base_project,
    project,
    labels=("Original Stillwater", "Raster GWI/RDII"),
    scenario_name="bwf_gwi_rdii",
    start=start,
    end=end,
)
